# Van de Vusse System Identification

Canonical Van de Vusse system-identification notebook. This workflow builds the nonlinear plant step tests, fits delayed FOPDT channels, and constructs a Python-only delay-absorbed discrete state-space model.

## User Config

In [ ]:
import os

from systems.vandevusse import get_vandevusse_notebook_defaults
from utils.notebook_setup import prepare_vandevusse_notebook_env, print_grouped_notebook_summary

NB = get_vandevusse_notebook_defaults("system_identification")
RUN_NEW_EXPERIMENTS = NB["run_new_experiments"]
SHOW_FOPDT_PLOTS = NB["show_fopdt_plots"]
SHOW_VALIDATION_PLOTS = NB["show_validation_plots"]
SAVE_METADATA_JSON = NB["save_metadata_json"]
DELAY_QUANTIZATION = NB["delay_quantization"]
VANDEVUSSE_DATA_DIR_OVERRIDE = NB["data_dir_override"]
VANDEVUSSE_RESULTS_DIR_OVERRIDE = NB["results_dir_override"]

REPO_ROOT, DATA_DIR, RESULT_DIR = prepare_vandevusse_notebook_env(
    data_dir_override=VANDEVUSSE_DATA_DIR_OVERRIDE,
    results_dir_override=VANDEVUSSE_RESULTS_DIR_OVERRIDE,
)
os.chdir(REPO_ROOT)


## Imports

In [ ]:
import pickle

import numpy as np

from systems.vandevusse import VANDEVUSSE_SYSTEM_METADATA
from systems.vandevusse.plant import build_vandevusse_system
from systems.vandevusse.system_id import (
    apply_deviation_form_scaled,
    build_vandevusse_identification_model,
    build_vandevusse_step_test_inputs,
    compute_vandevusse_min_max_states,
    run_vandevusse_step_test_experiment,
    save_vandevusse_identification_artifacts,
    scaling_min_max_factors,
    validate_vandevusse_identified_model,
)


## System Setup

In [ ]:
SYS = NB["system_setup"]
FIT_CFG = NB["fit"]
delta_t = SYS["delta_t_hours"]
system_params = SYS["system_params"].copy()
design_params = SYS["design_params"].copy()
ss_inputs = SYS["ss_inputs"].copy()
benchmark_state_seed = SYS["benchmark_state_seed"].copy()
input_bounds = {
    "u_min": SYS["input_bounds"]["u_min"].copy(),
    "u_max": SYS["input_bounds"]["u_max"].copy(),
}
step_tests = build_vandevusse_step_test_inputs(
    ss_inputs=ss_inputs,
    step_tests=NB["step_tests"],
    delta_t=delta_t,
    initial_hold_hours=NB["initial_hold_hours"],
    step_hold_hours=NB["step_hold_hours"],
    input_bounds=input_bounds,
)
plant = build_vandevusse_system(
    params=system_params,
    design_params=design_params,
    ss_inputs=ss_inputs,
    delta_t=delta_t,
    deviation_form=False,
)
steady_states = {
    "x_ss": np.asarray(plant.steady_trajectory, float).copy(),
    "ss_inputs": np.asarray(plant.ss_inputs, float).copy(),
    "y_ss": np.asarray(plant.y_ss, float).copy(),
}
solved_step_inputs = {
    cfg["name"]: (steady_states["ss_inputs"] + np.asarray(cfg["step_delta"], float)).tolist()
    for cfg in step_tests
}


## Resolved Summary

In [ ]:
print_grouped_notebook_summary(
    "Van de Vusse system-identification summary",
    {
        "Paths": {
            "Repo root": REPO_ROOT,
            "Data dir": DATA_DIR,
            "Results dir": RESULT_DIR,
        },
        "Execution flags": {
            "run_new_experiments": RUN_NEW_EXPERIMENTS,
            "show_fopdt_plots": SHOW_FOPDT_PLOTS,
            "show_validation_plots": SHOW_VALIDATION_PLOTS,
            "save_metadata_json": SAVE_METADATA_JSON,
            "delay_quantization": DELAY_QUANTIZATION,
        },
        "Benchmark setup": {
            "delta_t_hours": delta_t,
            "benchmark_state_seed": benchmark_state_seed.tolist(),
            "ss_inputs": steady_states["ss_inputs"].tolist(),
            "solved_x_ss": steady_states["x_ss"].tolist(),
            "solved_y_ss": steady_states["y_ss"].tolist(),
        },
        "Step tests": {
            cfg["name"]: solved_step_inputs[cfg["name"]]
            for cfg in step_tests
        },
        "Exports": {
            "system_dict": DATA_DIR / "system_dict.pickle",
            "scaling_factor": DATA_DIR / "scaling_factor.pickle",
            "min_max_states": DATA_DIR / "min_max_states.pickle",
            "metadata_pickle": DATA_DIR / "system_id_metadata.pickle",
            "metadata_json": DATA_DIR / "system_id_metadata.json",
            "validation_plot": RESULT_DIR / "vandevusse_identified_vs_nonlinear.png",
        },
    },
)


## Identification Workflow

In [ ]:
step_results_by_name = {}
if RUN_NEW_EXPERIMENTS:
    for step_cfg in step_tests:
        step_results_by_name[step_cfg["name"]] = run_vandevusse_step_test_experiment(
            step_cfg=step_cfg,
            system_params=system_params,
            design_params=design_params,
            ss_inputs=steady_states["ss_inputs"],
            delta_t=delta_t,
            data_dir=DATA_DIR,
            result_dir=RESULT_DIR,
            show_plot=False,
        )
else:
    print("RUN_NEW_EXPERIMENTS=False, reusing canonical VanDeVusse/Data CSV files.")

file_paths = {cfg["name"]: DATA_DIR / cfg["save_filename"] for cfg in step_tests}
missing_paths = [path for path in file_paths.values() if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing Van de Vusse step-test files: {missing_paths}")

print("Step-test CSV files:")
for name, path in file_paths.items():
    print(f"  {name:<20} -> {path}")


In [ ]:
data_min, data_max = scaling_min_max_factors(file_paths)
scaling_factor = {
    "min": np.asarray(data_min, float),
    "max": np.asarray(data_max, float),
}

deviation_dfs = apply_deviation_form_scaled(
    {"ss_inputs": steady_states["ss_inputs"], "y_ss": steady_states["y_ss"]},
    file_paths,
    data_min,
    data_max,
)

id_bundle = build_vandevusse_identification_model(
    deviation_dfs=deviation_dfs,
    step_tests=step_tests,
    delta_t=delta_t,
    pre_window_steps=FIT_CFG["pre_window_steps"],
    post_window_steps=FIT_CFG["post_window_steps"],
    plot=SHOW_FOPDT_PLOTS,
    quantization=DELAY_QUANTIZATION,
)
system_dict = id_bundle["system_dict"]

validation = validate_vandevusse_identified_model(
    system_dict=system_dict,
    deviation_dfs=deviation_dfs,
    step_tests=step_tests,
    delta_t=delta_t,
    result_dir=RESULT_DIR,
    show_plot=SHOW_VALIDATION_PLOTS,
)

if RUN_NEW_EXPERIMENTS:
    min_max_states = compute_vandevusse_min_max_states(step_results_by_name, steady_states)
else:
    min_max_path = DATA_DIR / "min_max_states.pickle"
    if not min_max_path.exists():
        raise FileNotFoundError(
            "RUN_NEW_EXPERIMENTS=False requires an existing VanDeVusse/Data/min_max_states.pickle."
        )
    with open(min_max_path, "rb") as handle:
        min_max_states = pickle.load(handle)

metadata = {
    "system_name": VANDEVUSSE_SYSTEM_METADATA["system_name"],
    "delta_t_hours": float(delta_t),
    "delay_quantization": DELAY_QUANTIZATION,
    "benchmark_state_seed": benchmark_state_seed,
    "ss_inputs": steady_states["ss_inputs"],
    "solved_x_ss": steady_states["x_ss"],
    "solved_y_ss": steady_states["y_ss"],
    "step_tests": step_tests,
    "file_paths": file_paths,
    "fit_results_by_test": id_bundle["fit_results_by_test"],
    "aggregated_fits": id_bundle["aggregated_fits"],
    "delay_steps": id_bundle["state_space"]["delay_steps"],
    "channel_specs": id_bundle["state_space"]["channel_specs"],
}

artifact_paths = save_vandevusse_identification_artifacts(
    repo_root=REPO_ROOT,
    data_dir=DATA_DIR,
    result_dir=RESULT_DIR,
    system_dict=system_dict,
    scaling_factor=scaling_factor,
    min_max_states=min_max_states,
    metadata=metadata,
    save_metadata_json=SAVE_METADATA_JSON,
)

print("Scaling min:", scaling_factor["min"])
print("Scaling max:", scaling_factor["max"])
print("Delay steps:\n", id_bundle["state_space"]["delay_steps"])
print("Saved artifacts:")
for key, path in artifact_paths.items():
    print(f"  {key:<16} -> {path}")

system_dict


## Plotting And Export

In [ ]:
print("Saved result figures:")
for path in sorted(RESULT_DIR.glob("*.png")):
    print(f"  {path.name}")
